# Graph-Flashback — Austin Gowalla (Kaggle)

Attach the Kaggle Dataset containing `gowalla_austin_enriched_195k.zip`, enable a T4 GPU, then run cells in order.


In [ ]:
%cd /kaggle/working
!rm -rf GeoGNNProject
!git clone --depth 1 --branch flashback-branch --single-branch https://github.com/klyuchnikova/GeoGNNProject.git
%cd /kaggle/working/GeoGNNProject
!git branch --show-current
!git log -1 --oneline


In [ ]:
!grep -v '^torch' requirements.txt > /tmp/requirements-kaggle.txt
!python -m pip install -q -r /tmp/requirements-kaggle.txt


In [ ]:
!python scripts/install_austin_dataset.py
!python scripts/verify_merge.py

import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## Optional smoke-test


In [ ]:
!python scripts/make_synthetic_data.py
!python -m flashback.pipeline --config configs/gowalla_smoke.yaml --stage all
!python -m pytest -q --basetemp=/kaggle/working/pytest_tmp


## Tuned Austin experiment


In [ ]:
CONFIG = "configs/gowalla_auto.yaml"


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage prepare


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage stkg


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage kge


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage graphs


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage train


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage analyze


## Inspect metrics and diagnostics


In [ ]:
from pathlib import Path
import json
import pandas as pd

for path in sorted(Path("artifacts/results").glob("*.json")):
    print("\nFILE:", path)
    print(json.dumps(json.loads(path.read_text()), indent=2))

comparison = Path("artifacts/results/metrics_comparison.csv")
if comparison.exists():
    display(pd.read_csv(comparison))

for path in [Path("data/kge/stkg_manifest.json"), Path("data/kge/transe_diagnostics.json"), Path("data/graphs/graph_manifest.json")]:
    if path.exists():
        print("\nFILE:", path)
        print(json.dumps(json.loads(path.read_text()), indent=2))


## Save compact results archive


In [ ]:
from pathlib import Path
import zipfile

root = Path("/kaggle/working/GeoGNNProject")
output = Path("/kaggle/working/graph_flashback_austin_results.zip")
include = [
    root / "artifacts",
    root / "data/processed",
    root / "data/kge/stkg_manifest.json",
    root / "data/kge/transe_history.json",
    root / "data/kge/transe_diagnostics.json",
    root / "data/graphs/graph_manifest.json",
    root / "configs/gowalla_auto.yaml",
]
excluded = {".pt", ".pth", ".pkl", ".npz", ".npy"}
with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in include:
        if not path.exists():
            continue
        files = [path] if path.is_file() else path.rglob("*")
        for file in files:
            if file.is_file() and file.suffix.lower() not in excluded:
                archive.write(file, file.relative_to(root))
print(output, f"{output.stat().st_size / 1024**2:.2f} MB")
